# XTTS Fine-tuned Model Inference

This notebook runs inference using a fine-tuned **XTTS v2** model.

The general flow is:

1. Mount Google Drive so the model files and speaker audio can be accessed.
2. Set up the Colab environment and install the required dependencies.
3. Clone the XTTS fine-tuning repository.
4. Select the checkpoint, config file, vocabulary file, speaker audio, and input text.
5. Run `run_inference.py` to generate a `.wav` audio file.
6. Play the generated audio inside the notebook.

> **Note:** This notebook is intended for Google Colab. Some cells use Colab-specific commands such as `drive.mount()` and shell commands beginning with `!`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Set Up Python 3.10

In [ ]:
# 1. Install Python 3.10
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-distutils -y

# 2. Add Python 3.10 to the update-alternatives list
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1

# 3. Configure the default (you'll need to input the selection number if prompted)
# In Colab, you can often skip the manual prompt by setting priority,
# but this command ensures the link is created.
!sudo update-alternatives --set python3 /usr/bin/python3.10

# 4. Reinstall pip for the new version
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!python3 get-pip.py --force-reinstall

# 5. Check the version
!python --version

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,701 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,297 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,090 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,959 kB]
Get:13 https://r2u.stat.illinois

## 3. Clone or Open the XTTS Repository

In [ ]:
import os

REPO_DIR = "/content/XTTSv2-Finetuning-for-New-Languages"

if not os.path.exists(REPO_DIR):
    # !git clone https://github.com/nguyenhoanganh2002/XTTSv2-Finetuning-for-New-Languages.git {REPO_DIR}
    !git clone https://github.com/Fabzamm/XTTSv2-Finetuning-for-New-Languages.git {REPO_DIR}

%cd {REPO_DIR}

Cloning into '/content/XTTSv2-Finetuning-for-New-Languages'...
remote: Enumerating objects: 726, done.
remote: Counting objects: 100% (316/316), done.
remote: Compressing objects: 100% (218/218), done.
remote: Total 726 (delta 137), reused 98 (delta 98), pack-reused 410 (from 3)
Receiving objects: 100% (726/726), 2.12 MiB | 6.81 MiB/s, done.
Resolving deltas: 100% (213/213), done.
/content/XTTSv2-Finetuning-for-New-Languages


## 4. Remove Possible `blinker` Conflicts


In [ ]:
# Force-remove pre-installed 'blinker' to avoid version conflicts in Colab
!find /usr/lib/python3 -name "blinker*" -exec rm -rf {} + 2>/dev/null
!find /usr/local/lib/python3.10 -name "blinker*" -exec rm -rf {} + 2>/dev/null

## 5. Install Extra Audio Dependency


In [ ]:
!pip install torchcodec==0.10.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.0 MB/s  0:00:00


## 6. Install the Project Requirements

In [ ]:
# Took around 5-6 minutes

%cd /content/XTTSv2-Finetuning-for-New-Languages
!pip install -r requirements.txt

/content/XTTSv2-Finetuning-for-New-Languages
Ignoring numpy: markers 'python_version > "3.10"' don't match your environment
Ignoring numba: markers 'python_version < "3.9"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 44.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 91.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing meta

## 7. Install IPython Audio Support


In [ ]:
!pip install ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.8/831.8 kB 8.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 24.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [ipython]


## 8. Configure and Run XTTS Inference

This is the main inference cell.

Here you choose:

- `TEXT`: the Maltese sentence to synthesise.
- `CHECKPOINT`: the fine-tuned model weights.
- `CONFIG`: the model configuration file.
- `VOCAB`: the tokenizer vocabulary file.
- `SPEAKER`: the reference speaker audio used for voice cloning.
- `OUTPUT`: where the generated `.wav` file will be saved.
- `SPEED`: speech speed multiplier.

Only one checkpoint setup should be active at a time.  
The other checkpoint blocks are kept as commented alternatives so you can quickly switch between experiments.

In [ ]:
# Takes about 2-6 minutes

TEXT = "Din hija sentenza biex nara kif jaħdem il-mudell."

# # Checkpoints_for_updated_vocab_+_5_Epochs_DVAE (10 epochs)
# CHECKPOINT = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_updated_vocab_+_5_Epochs_DVAE/Epochs_10_11_GPT_XTTS_FT-March-20-2026_02+30PM-8ddb4db/checkpoint_44430.pth"
# CONFIG     = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_updated_vocab_+_5_Epochs_DVAE/Epochs_10_11_GPT_XTTS_FT-March-20-2026_02+30PM-8ddb4db/config.json"
# VOCAB      = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Model files for Updated Vocab + 5 Epochs DVAE (XTTS_v2.0_original_model_files)/vocab.json"

# # GPT Only
# CHECKPOINT = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_GPT_only/9_10_Epochs_GPT_XTTS_FT-March-09-2026_03+18PM-a84436e/checkpoint_44420.pth"
# CONFIG     = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_GPT_only/9_10_Epochs_GPT_XTTS_FT-March-09-2026_03+18PM-a84436e/config.json"
# VOCAB      = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Model files for GPT only (XTTS_v2.0_original_model_files)/vocab.json"
# # VOCAB = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/XTTS_v2.0_original_model_files/vocab.json"

# # DVAE 5 Epochs
# CHECKPOINT = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_DVAE_5_Epochs/10_11_Epochs_GPT_XTTS_FT-March-29-2026_10+17AM-f78149c/checkpoint_44430.pth"
# CONFIG     = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_DVAE_5_Epochs/10_11_Epochs_GPT_XTTS_FT-March-29-2026_10+17AM-f78149c/config.json"
# VOCAB      = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Model files for DVAE 5 Epochs (XTTS_v2.0_original_model_files)/vocab.json"

# # DVAE 10 Epochs
# CHECKPOINT = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_DVAE_10_Epochs/10_Epochs_GPT_XTTS_FT-April-07-2026_01+18PM-4105e37/best_model_44431.pth"
# CONFIG     = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_DVAE_10_Epochs/10_Epochs_GPT_XTTS_FT-April-07-2026_01+18PM-4105e37/config.json"
# VOCAB      = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Model files for DVAE 10 Epochs (XTTS_v2.0_original_model_files)/vocab.json"

# Adding Korpus
CHECKPOINT = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_adding_Korpus/10_Epochs_GPT_XTTS_FT-April-03-2026_09+10AM-4105e37/best_model_44433.pth"
CONFIG     = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_adding_Korpus/10_Epochs_GPT_XTTS_FT-April-03-2026_09+10AM-4105e37/config.json"
VOCAB      = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Model files for Adding Korpus (XTTS_v2.0_original_model_files)/vocab.json"

# # Adding_Korpus_Full
# CHECKPOINT = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_adding_Korpus_Full/10_Epochs_GPT_XTTS_FT-May-05-2026_01+40PM-cb1aef8/best_model_44431.pth"
# CONFIG     = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_adding_Korpus_Full/10_Epochs_GPT_XTTS_FT-May-05-2026_01+40PM-cb1aef8/config.json"
# VOCAB      = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Model files for Adding_Korpus_Full (XTTS_v2.0_original_model_files)/vocab.json"

# # Vocab 2000
# CHECKPOINT = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_vocab_2000/8_9_10_Epochs_GPT_XTTS_FT-March-28-2026_01+15PM-f78149c/best_model_44434.pth"
# CONFIG     = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_vocab_2000/8_9_10_Epochs_GPT_XTTS_FT-March-28-2026_01+15PM-f78149c/config.json"
# VOCAB      = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Model files for vocab 2000 (XTTS_v2.0_original_model_files)/vocab.json"

# # Vocab 750
# CHECKPOINT = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_vocab_750/9_10_Epochs_GPT_XTTS_FT-April-11-2026_08+14AM-f6b62e9/checkpoint_44430.pth"
# CONFIG     = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_vocab_750/9_10_Epochs_GPT_XTTS_FT-April-11-2026_08+14AM-f6b62e9/config.json"
# VOCAB      = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Model files for vocab 750 (XTTS_v2.0_original_model_files)/vocab.json"

# # Vocab 500
# CHECKPOINT = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_vocab_500/10_Epochs_GPT_XTTS_FT-April-15-2026_01+46PM-7048429/best_model_44432.pth"
# CONFIG     = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_vocab_500/10_Epochs_GPT_XTTS_FT-April-15-2026_01+46PM-7048429/config.json"
# VOCAB      = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Model files for vocab 500 (XTTS_v2.0_original_model_files)/vocab.json"

# # Vocab 500 & DVAE 10 Epochs
# CHECKPOINT = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_vocab_500_and_DVAE_10_Epochs/9_10_Epochs_GPT_XTTS_FT-April-20-2026_02+32PM-7048429/best_model_44433.pth"
# CONFIG = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_vocab_500_and_DVAE_10_Epochs/9_10_Epochs_GPT_XTTS_FT-April-20-2026_02+32PM-7048429/config.json"
# VOCAB = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Model files for vocab 500 and DVAE 10 Epochs (XTTS_v2.0_original_model_files)/vocab.json"


SPEAKER = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/Voices/voice_mt.wav"
OUTPUT = "/content/drive/MyDrive/FYP/Audio.wav"
SPEED = 1

!python run_inference.py \
    --checkpoint "{CHECKPOINT}" \
    --config "{CONFIG}" \
    --vocab "{VOCAB}" \
    --speaker_audio "{SPEAKER}" \
    --text "{TEXT}" \
    --output_path "{OUTPUT}" \
    --speed {SPEED}

GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
✓ Model loaded!
  0% 0/1 [00:00<?, ?it/s]Original text : L-ittri huma: 'għ', 'ħ', u 'ċ'
Token IDs     : [6681, 25, 8, 2, 60, 1056, 2, 1874, 14, 2, 11, 2, 4, 2, 6

## 9. Play the Generated Audio


In [ ]:
# ── Play Output ───────────────────────────────────────────────────────────────
from IPython.display import Audio
Audio(OUTPUT)